# NspireLM: distill the calculator chat model

Distills SmolLM2-135M-Instruct into the 9-layer, 16k-vocabulary, int4 student that fits the TI-Nspire CX II.
**Runtime → Change runtime type → GPU**, then *Run all*.
Checkpoints go to Google Drive (`MyDrive/nspirelm/`). If Colab disconnects, run all again and it resumes.
When it finishes, download `student.bin`, `student.bin.json` and `student.bin.qat.npz` from the last cell's folder.

In [ ]:
BRANCH = "claude/confident-planck-cw3n5u"  # or main once merged
STEPS = 12000   # ~200M tokens. On a free T4 use 6000 (about 2 h)
RUN = "s9"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists("NspireLM"):
    !git clone -q -b $BRANCH https://github.com/TypeThe0ry/NspireLM
!pip -q install -U transformers pyarrow huggingface_hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/nspirelm/{RUN}"
os.makedirs(OUT, exist_ok=True)
print(OUT)

In [ ]:
%cd /content/NspireLM
!git pull -q
# keep.npy / layers.json are committed (train/artifacts); regenerate with build_vocab.py / select_layers.py if you change sizes
!python3 train/distill.py --out $OUT --layers train/artifacts/layers.json --keep train/artifacts/keep.npy --steps $STEPS --batch 32 --seq 512

In [ ]:
!python3 train/export.py $OUT/student.pt $OUT/student.bin
!ls -la $OUT